In [1]:
import pandas as pd
import altair as alt

url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/licenses_fall2022.csv"
df = pd.read_csv(url)

df['Effective Date'] = pd.to_datetime(df['Effective Date'], errors='coerce')
df['Year'] = df['Effective Date'].dt.year

df_filtered = df[(df['Year'] > 1970) & (df['Year'] < 2023)].copy()

status_counts = df_filtered.groupby(['Year', 'License Status']).size().reset_index(name='Count')

top_cities = df_filtered['City'].value_counts().head(15).index.tolist()
df_top_cities = df_filtered[df_filtered['City'].isin(top_cities)]
city_status_counts = df_top_cities.groupby(['City', 'License Status']).size().reset_index(name='Count')

status_list = status_counts['License Status'].dropna().unique().tolist()
dropdown = alt.binding_select(options=[None] + status_list,
                              labels=['All Statuses'] + status_list,
                              name='Select Status: ')
selection = alt.selection_point(fields=['License Status'], bind=dropdown)

timeline = alt.Chart(status_counts).mark_bar(size=10).encode(
    x=alt.X('Year:O', title='Effective Year', axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('Count:Q', title='Number of Licenses', stack='zero'),
    color=alt.condition(selection, alt.Color('License Status:N', scale=alt.Scale(scheme='set2')), alt.value('lightgray')),
    opacity=alt.condition(selection, alt.value(1.0), alt.value(0.2)),
    tooltip=['Year:O', 'License Status:N', 'Count:Q']
).add_params(
    selection
).properties(
    width=600, height=300,
    title='License Issuance Trend by Status'
)

heatmap = alt.Chart(city_status_counts).mark_rect().encode(
    x=alt.X('License Status:N', title='License Status', axis=alt.Axis(labelAngle=-45)),
    y=alt.Y('City:N', sort='-color', title='City Name'),
    color=alt.Color('Count:Q', scale=alt.Scale(scheme='blues'), title='Total Licenses'),
    tooltip=['City:N', 'License Status:N', 'Count:Q']
).transform_filter(
    selection
).properties(
    width=600, height=350,
    title='License Density by City and Status'
)

hw_final_chart = alt.vconcat(timeline, heatmap, spacing=40).resolve_scale(color='independent')
hw_final_chart.save('licenses_chart_hw5.1.json')
hw_final_chart

alt.VConcatChart(...)